In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/march-machine-learning-mania-2025/Conferences.csv
/kaggle/input/march-machine-learning-mania-2025/SeedBenchmarkStage1.csv
/kaggle/input/march-machine-learning-mania-2025/WNCAATourneyDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2025/WRegularSeasonCompactResults.csv
/kaggle/input/march-machine-learning-mania-2025/MNCAATourneySeedRoundSlots.csv
/kaggle/input/march-machine-learning-mania-2025/MRegularSeasonDetailedResults.csv
/kaggle/input/march-machine-learning-mania-2025/MNCAATourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2025/MGameCities.csv
/kaggle/input/march-machine-learning-mania-2025/WSecondaryTourneyCompactResults.csv
/kaggle/input/march-machine-learning-mania-2025/WGameCities.csv
/kaggle/input/march-machine-learning-mania-2025/MSeasons.csv
/kaggle/input/march-machine-learning-mania-2025/WNCAATourneySlots.csv
/kaggle/input/march-machine-learning-mania-2025/MSecondaryTourneyTeams.csv
/kaggle/input/march-machine-learning-mania-20

In [5]:
# 导入所需的库
print("开始导入模块...")
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple
import joblib
import xgboost as xgb
import torch
import lightgbm as lgb
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
import gc
from lightgbm import LGBMClassifier
import optuna
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 设置随机种子
np.random.seed(42)

print("模块导入完成")

开始导入模块...
模块导入完成


In [6]:
print("\n=== 开始加载数据 ===")

# 加载数据
print("加载锦标赛结果...")
tournament_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2025/MNCAATourneyDetailedResults.csv")
print(f"加载了 {len(tournament_results)} 场锦标赛比赛")

print("\n加载种子排名数据...")
tournament_seeds = pd.read_csv("/kaggle/input/march-machine-learning-mania-2025/MNCAATourneySeeds.csv")
print(f"加载了 {len(tournament_seeds)} 条种子排名数据")

print("\n加载常规赛结果...")
regular_season_results = pd.read_csv("/kaggle/input/march-machine-learning-mania-2025/MRegularSeasonDetailedResults.csv")
print(f"加载了 {len(regular_season_results)} 场常规赛比赛")

print("\n加载球队数据...")
teams = pd.read_csv("/kaggle/input/march-machine-learning-mania-2025/MTeams.csv")
print(f"加载了 {len(teams)} 支球队的数据")

print("\n加载赛季信息...")
seasons = pd.read_csv("/kaggle/input/march-machine-learning-mania-2025/MSeasons.csv")
print(f"加载了 {len(seasons)} 个赛季的信息")

print("\n加载Massey排名数据...")
massey_ordinals = pd.read_csv("/kaggle/input/march-machine-learning-mania-2025/MMasseyOrdinals.csv")
print(f"加载了 {len(massey_ordinals)} 条Massey排名数据")


=== 开始加载数据 ===
加载锦标赛结果...
加载了 1382 场锦标赛比赛

加载种子排名数据...
加载了 2558 条种子排名数据

加载常规赛结果...
加载了 117748 场常规赛比赛

加载球队数据...
加载了 380 支球队的数据

加载赛季信息...
加载了 41 个赛季的信息

加载Massey排名数据...
加载了 5489117 条Massey排名数据


In [7]:
print("\n=== 计算基础统计数据 ===")

def calculate_basic_team_stats(season_games, team_id, current_game_date):
    """计算球队基础统计数据，只使用当前比赛日期之前的数据"""
    team_games = season_games[
        ((season_games['WTeamID'] == team_id) | 
         (season_games['LTeamID'] == team_id)) &
        (season_games['DayNum'] < current_game_date)  # 只使用当前比赛之前的数据
    ].copy()
    
    if len(team_games) == 0:
        # 返回默认值的DataFrame
        return pd.DataFrame([{
            'TeamID': team_id,
            'WinRate': 0.5,
            'TotalGames': 0,
            'Wins': 0,
            'Losses': 0,
            'WinStreak': 0,
            'PointsPerGame': 0,
            'PointsAllowed': 0,
            'OffensiveRating': 0,
            'DefensiveRating': 0,
            'NetRating': 0,
            'PaceAdjustedOffRating': 0,
            'PaceAdjustedDefRating': 0,
            'FieldGoalPct': 0,
            'ThreePointPct': 0,
            'FreeThrowPct': 0,
            'ReboundRate': 0,
            'AssistRate': 0,
            'TurnoverRate': 0,
            'BlockRate': 0,
            'StealRate': 0
        }])
    
    # 获取球队比赛
    team_wins = team_games[team_games['WTeamID'] == team_id]
    team_losses = team_games[team_games['LTeamID'] == team_id]
    total_games = len(team_wins) + len(team_losses)
    
    # 基本统计
    wins = len(team_wins)
    losses = len(team_losses)
    win_rate = wins / total_games if total_games > 0 else 0
    
    # 得分统计
    points_scored = team_wins['WScore'].sum() + team_losses['LScore'].sum()
    points_allowed = team_wins['LScore'].sum() + team_losses['WScore'].sum()
    points_per_game = points_scored / total_games if total_games > 0 else 0
    points_allowed_per_game = points_allowed / total_games if total_games > 0 else 0
    
    # 计算连胜
    games_chronological = pd.concat([
        team_wins[['Season', 'DayNum', 'WTeamID']].assign(won=1),
        team_losses[['Season', 'DayNum', 'LTeamID']].assign(won=0)
    ]).sort_values('DayNum')
    current_streak = 0
    for won in games_chronological['won']:
        if won:
            current_streak = max(0, current_streak + 1)
        else:
            current_streak = min(0, current_streak - 1)
    
    # 计算投篮命中率
    fg_attempts = team_wins['WFGA'].sum() + team_losses['LFGA'].sum()
    fg_made = team_wins['WFGM'].sum() + team_losses['LFGM'].sum()
    fg_pct = fg_made / fg_attempts if fg_attempts > 0 else 0
    
    three_attempts = team_wins['WFGA3'].sum() + team_losses['LFGA3'].sum()
    three_made = team_wins['WFGM3'].sum() + team_losses['LFGM3'].sum()
    three_pct = three_made / three_attempts if three_attempts > 0 else 0
    
    ft_attempts = team_wins['WFTA'].sum() + team_losses['LFTA'].sum()
    ft_made = team_wins['WFTM'].sum() + team_losses['LFTM'].sum()
    ft_pct = ft_made / ft_attempts if ft_attempts > 0 else 0
    
    # 计算进阶统计
    total_possessions = (
        team_wins['WFGA'].sum() - team_wins['WOR'].sum() + 
        team_wins['WTO'].sum() + 0.44 * team_wins['WFTA'].sum() +
        team_losses['LFGA'].sum() - team_losses['LOR'].sum() + 
        team_losses['LTO'].sum() + 0.44 * team_losses['LFTA'].sum()
    )
    
    offensive_rating = 100 * points_scored / total_possessions if total_possessions > 0 else 0
    defensive_rating = 100 * points_allowed / total_possessions if total_possessions > 0 else 0
    net_rating = offensive_rating - defensive_rating
    
    # 计算节奏
    pace = 40 * total_possessions / (total_games * 40) if total_games > 0 else 0  # 假设每场比赛40分钟
    pace_adjusted_off = offensive_rating * (pace / 70) if pace > 0 else 0  # 标准化到70回合
    pace_adjusted_def = defensive_rating * (70 / pace) if pace > 0 else 0
    
    # 计算其他高级数据
    total_rebounds = (
        team_wins['WOR'].sum() + team_wins['WDR'].sum() +
        team_losses['LOR'].sum() + team_losses['LDR'].sum()
    )
    total_opp_rebounds = (
        team_wins['LOR'].sum() + team_wins['LDR'].sum() +
        team_losses['WOR'].sum() + team_losses['WDR'].sum()
    )
    rebound_rate = total_rebounds / (total_rebounds + total_opp_rebounds) if (total_rebounds + total_opp_rebounds) > 0 else 0
    
    total_assists = team_wins['WAst'].sum() + team_losses['LAst'].sum()
    total_fga = team_wins['WFGA'].sum() + team_losses['LFGA'].sum()
    assist_rate = total_assists / total_fga if total_fga > 0 else 0
    
    total_turnovers = team_wins['WTO'].sum() + team_losses['LTO'].sum()
    turnover_rate = total_turnovers / total_possessions if total_possessions > 0 else 0
    
    total_blocks = team_wins['WBlk'].sum() + team_losses['LBlk'].sum()
    total_opp_fga = team_wins['LFGA'].sum() + team_losses['WFGA'].sum()
    block_rate = total_blocks / total_opp_fga if total_opp_fga > 0 else 0
    
    total_steals = team_wins['WStl'].sum() + team_losses['LStl'].sum()
    steal_rate = total_steals / total_possessions if total_possessions > 0 else 0
    
    # 返回DataFrame格式的统计数据
    return pd.DataFrame([{
        'TeamID': team_id,
        'WinRate': win_rate,
        'TotalGames': total_games,
        'Wins': wins,
        'Losses': losses,
        'WinStreak': current_streak,
        'PointsPerGame': points_per_game,
        'PointsAllowed': points_allowed_per_game,
        'OffensiveRating': offensive_rating,
        'DefensiveRating': defensive_rating,
        'NetRating': net_rating,
        'PaceAdjustedOffRating': pace_adjusted_off,
        'PaceAdjustedDefRating': pace_adjusted_def,
        'FieldGoalPct': fg_pct,
        'ThreePointPct': three_pct,
        'FreeThrowPct': ft_pct,
        'ReboundRate': rebound_rate,
        'AssistRate': assist_rate,
        'TurnoverRate': turnover_rate,
        'BlockRate': block_rate,
        'StealRate': steal_rate
    }])


=== 计算基础统计数据 ===


In [8]:
print("\n=== 计算四因素 ===")

def calculate_four_factors(season_games, team_id, current_game_date):
    """计算四因素,只使用当前比赛之前的数据"""
    team_games = season_games[
        ((season_games['WTeamID'] == team_id) |
         (season_games['LTeamID'] == team_id)) &
        (season_games['DayNum'] < current_game_date)
    ]
    
    if len(team_games) == 0:
        return pd.DataFrame()
        
    # 获取球队比赛
    team_wins = team_games[team_games['WTeamID'] == team_id]
    team_losses = team_games[team_games['LTeamID'] == team_id]
    
    # 计算进攻回合
    win_possessions = (
        team_wins['WFGA'] - team_wins['WOR'] + 
        team_wins['WTO'] + 0.44 * team_wins['WFTA']
    )
    loss_possessions = (
        team_losses['LFGA'] - team_losses['LOR'] + 
        team_losses['LTO'] + 0.44 * team_losses['LFTA']
    )
    total_possessions = sum(win_possessions) + sum(loss_possessions)
    
    # 计算得分
    win_points = team_wins['WScore'].sum()
    loss_points = team_losses['LScore'].sum()
    total_points = win_points + loss_points
    
    # 计算进攻效率
    offensive_rating = (
        100 * total_points / total_possessions 
        if total_possessions > 0 else 0
    )
    
    # 计算防守效率
    opp_win_points = team_wins['LScore'].sum()
    opp_loss_points = team_losses['WScore'].sum()
    defensive_rating = (
        100 * (opp_win_points + opp_loss_points) / total_possessions
        if total_possessions > 0 else 0
    )
    
    # 计算篮板率
    total_rebounds = (
        team_wins['WOR'].sum() + team_wins['WDR'].sum() +
        team_losses['LOR'].sum() + team_losses['LDR'].sum()
    )
    total_opp_rebounds = (
        team_wins['LOR'].sum() + team_wins['LDR'].sum() +
        team_losses['WOR'].sum() + team_losses['WDR'].sum()
    )
    rebound_rate = (
        total_rebounds / (total_rebounds + total_opp_rebounds)
        if (total_rebounds + total_opp_rebounds) > 0 else 0
    )
    
    # 计算失误率
    total_turnovers = team_wins['WTO'].sum() + team_losses['LTO'].sum()
    turnover_rate = (
        total_turnovers / total_possessions
        if total_possessions > 0 else 0
    )
    
    # 计算罚球率
    total_fta = team_wins['WFTA'].sum() + team_losses['LFTA'].sum()
    total_fga = team_wins['WFGA'].sum() + team_losses['LFGA'].sum()
    free_throw_rate = total_fta / total_fga if total_fga > 0 else 0
    
    return pd.DataFrame([{
        'TeamID': team_id,
        'OffensiveRating': offensive_rating,
        'DefensiveRating': defensive_rating,
        'ReboundRate': rebound_rate,
        'TurnoverRate': turnover_rate,
        'FreeThrowRate': free_throw_rate
    }])


=== 计算四因素 ===


In [9]:
print("\n=== 计算高级统计和动量特征 ===")

def calculate_momentum_features(season_games, team_id, current_game_date):
    """计算球队的动量特征，只使用当前比赛日期之前的数据"""
    team_games = season_games[
        ((season_games['WTeamID'] == team_id) |
         (season_games['LTeamID'] == team_id)) &
        (season_games['DayNum'] < current_game_date)  # 只使用当前比赛之前的数据
    ].sort_values('DayNum', ascending=False)  # 按时间倒序排列
    
    if len(team_games) == 0:
        return {
            'last_5_win_rate': 0.5,
            'last_10_win_rate': 0.5,
            'last_15_win_rate': 0.5
        }
    
    # 添加是否获胜的标记
    team_games['IsWin'] = (team_games['WTeamID'] == team_id).astype(int)
    
    # 计算不同窗口的胜率
    momentum_features = {}
    for window in [5, 10, 15]:
        recent_games = team_games.head(window)  # 获取最近的n场比赛
        if len(recent_games) > 0:
            win_rate = recent_games['IsWin'].mean()
        else:
            win_rate = 0.5
        momentum_features[f'last_{window}_win_rate'] = win_rate
    
    return momentum_features
def calculate_efficiency_metrics(season_games, team_id, current_game_date):
    """计算效率指标，只使用当前比赛日期之前的数据"""
    team_games = season_games[
        ((season_games['WTeamID'] == team_id) |
         (season_games['LTeamID'] == team_id)) &
        (season_games['DayNum'] < current_game_date)
    ]
    
    # 初始化默认值
    metrics = {
        'offensive_efficiency': 0,
        'pace': 0
    }
    
    if len(team_games) == 0:
        return metrics
    
    total_points = 0
    total_possessions = 0
    
    for _, game in team_games.iterrows():
        if game['WTeamID'] == team_id:
            points = game['WScore']
            possessions = (
                game['WFGA'] - game['WOR'] +
                game['WTO'] + 0.44 * game['WFTA']
            )
        else:
            points = game['LScore']
            possessions = (
                game['LFGA'] - game['LOR'] +
                game['LTO'] + 0.44 * game['LFTA']
            )
        
        total_points += points
        total_possessions += possessions
    
    if total_possessions > 0:
        metrics['offensive_efficiency'] = 100 * total_points / total_possessions
        metrics['pace'] = total_possessions / len(team_games)
    
    return metrics

def calculate_consistency_rating(season_games, team_id, current_game_date):
    """计算球队表现一致性评分,只使用当前比赛之前的数据"""
    team_games = season_games[
        ((season_games['WTeamID'] == team_id) |
         (season_games['LTeamID'] == team_id)) &
        (season_games['DayNum'] < current_game_date)  # 只使用当前比赛之前的数据
    ]
    
    if len(team_games) == 0:
        return 0
    
    scores = []
    for _, game in team_games.iterrows():
        if game['WTeamID'] == team_id:
            scores.append(game['WScore'])
        else:
            scores.append(game['LScore'])
    
    # 计算得分标准差（越低越稳定）
    score_std = np.std(scores)
    max_std = 30  # 假设的最大标准差
    
    # 转换为0-1的评分（1表示最稳定）
    consistency = 1 - (score_std / max_std)
    return max(0, min(1, consistency))


=== 计算高级统计和动量特征 ===


In [10]:
print("\n=== 创建对抗特征 ===")
def create_matchup_features(team1_stats, team2_stats, season_games, team1_id, team2_id, current_game_date):
    """创建对抗特征"""
    if team1_stats.empty or team2_stats.empty:
        return None
    
    # 基础特征差异和比率
    feature_dict = {}
    base_cols = [
        'WinRate', 'PointsPerGame', 'PointsAllowed', 'OffensiveRating',
        'DefensiveRating', 'NetRating', 'PaceAdjustedOffRating',
        'PaceAdjustedDefRating', 'FieldGoalPct', 'ThreePointPct',
        'FreeThrowPct', 'ReboundRate', 'AssistRate', 'TurnoverRate',
        'BlockRate', 'StealRate'
    ]
    
    for col in base_cols:
        if col in team1_stats.columns and col in team2_stats.columns:
            feature_dict[f'{col}_diff'] = float(team1_stats[col].iloc[0] - team2_stats[col].iloc[0])
            feature_dict[f'{col}_ratio'] = float(team1_stats[col].iloc[0] / (team2_stats[col].iloc[0] + 1e-6))
    
    # 添加动量特征
    team1_momentum = calculate_momentum_features(season_games, team1_id, current_game_date)
    team2_momentum = calculate_momentum_features(season_games, team2_id, current_game_date)
    
    for key in team1_momentum:
        feature_dict[f'{key}_diff'] = team1_momentum[key] - team2_momentum[key]
        feature_dict[f'{key}_ratio'] = team1_momentum[key] / (team2_momentum[key] + 1e-6)
    
    # 添加效率指标
    team1_efficiency = calculate_efficiency_metrics(season_games, team1_id, current_game_date)
    team2_efficiency = calculate_efficiency_metrics(season_games, team2_id, current_game_date)
    
    efficiency_metrics = ['offensive_efficiency', 'pace']
    for key in efficiency_metrics:
        if key in team1_efficiency and key in team2_efficiency:
            feature_dict[f'{key}_diff'] = team1_efficiency[key] - team2_efficiency[key]
            feature_dict[f'{key}_ratio'] = team1_efficiency[key] / (team2_efficiency[key] + 1e-6)
    
    # 添加一致性评分
    team1_consistency = calculate_consistency_rating(season_games, team1_id, current_game_date)
    team2_consistency = calculate_consistency_rating(season_games, team2_id, current_game_date)
    feature_dict['consistency_diff'] = team1_consistency - team2_consistency
    feature_dict['consistency_ratio'] = team1_consistency / (team2_consistency + 1e-6)
    
    # 重要修改：直接返回特征字典，而不是DataFrame
    return feature_dict


=== 创建对抗特征 ===


In [11]:
print("\n=== 生成历史交手特征 ===")
def add_historical_matchup_features(matchup_features, all_games, team1_id, team2_id, current_season, current_game_date, lookback_years=5):
    """添加历史交手特征"""
    # 只使用当前比赛之前的数据
    historical_matchups = all_games[
        (
            # 之前赛季的数据
            ((all_games['Season'] >= current_season - lookback_years) &
             (all_games['Season'] < current_season)) |
            # 当前赛季早于当前比赛的数据
            ((all_games['Season'] == current_season) &
             (all_games['DayNum'] < current_game_date))
        ) &
        # 筛选两队之间的交手记录
        (
            ((all_games['WTeamID'] == team1_id) & 
             (all_games['LTeamID'] == team2_id)) |
            ((all_games['WTeamID'] == team2_id) & 
             (all_games['LTeamID'] == team1_id))
        )
    ]
    
    if len(historical_matchups) == 0:
        # 直接修改字典而不是返回DataFrame
        matchup_features['historical_win_rate'] = 0.5
        matchup_features['total_matchups'] = 0
        matchup_features['recent_matchup_win_rate'] = 0.5
        matchup_features['head_to_head_dominance'] = 0
        return matchup_features
    
    # 计算历史交手统计
    team1_wins = historical_matchups[historical_matchups['WTeamID'] == team1_id]
    total_games = len(historical_matchups)
    
    # 总体胜率
    win_rate = len(team1_wins) / total_games if total_games > 0 else 0.5
    matchup_features['historical_win_rate'] = win_rate
    matchup_features['total_matchups'] = total_games
    
    # 近期交手胜率（最近3场）
    recent_matchups = historical_matchups.sort_values('DayNum', ascending=False).head(3)
    recent_wins = len(recent_matchups[recent_matchups['WTeamID'] == team1_id])
    matchup_features['recent_matchup_win_rate'] = (
        recent_wins / len(recent_matchups)
        if len(recent_matchups) > 0 else 0.5
    )
    
    # 计算支配度（考虑比分差距）
    score_diffs = []
    for _, game in historical_matchups.iterrows():
        if game['WTeamID'] == team1_id:
            score_diffs.append(game['WScore'] - game['LScore'])
        else:
            score_diffs.append(game['LScore'] - game['WScore'])
    
    matchup_features['head_to_head_dominance'] = np.mean(score_diffs) if score_diffs else 0
    
    return matchup_features


=== 生成历史交手特征 ===


In [12]:
print("\n=== 生成种子排名特征===")
def add_tournament_seed_features(
    matchup_df, 
    tournament_seeds, 
    team1_id, 
    team2_id, 
    current_season
):
    """添加种子排名特征，只使用历史数据"""
    # 初始化默认值
    default_seed_features = {
        'team1_historical_seed': 8.0,
        'team2_historical_seed': 8.0,
        'team1_seed_trend': 0.0,
        'team2_seed_trend': 0.0,
        'historical_seed_diff': 0.0,
        'historical_seed_ratio': 1.0,
        'recent_seed_diff': 0.0,
        'recent_seed_ratio': 1.0
    }
    
    # 只使用历史数据
    historical_seeds = tournament_seeds[tournament_seeds['Season'] < current_season].copy()
    
    if len(historical_seeds) == 0:
        for key, value in default_seed_features.items():
            matchup_df[key] = value
        return matchup_df
    
    def get_team_seed_history(team_id):
        """获取球队的种子历史"""
        # 创建数据的副本
        team_seeds = historical_seeds[historical_seeds['TeamID'] == team_id].copy()
        if len(team_seeds) == 0:
            return None
        
        # 使用 .loc 进行赋值
        team_seeds.loc[:, 'SeedValue'] = team_seeds['Seed'].apply(lambda x: int(x[1:3]))
        
        # 计算历史平均种子
        avg_seed = team_seeds['SeedValue'].mean()
        
        # 计算种子趋势（使用最近3年的数据）
        recent_seeds = team_seeds.sort_values('Season', ascending=False).head(3)
        if len(recent_seeds) >= 2:
            seasons = recent_seeds['Season'].values
            seed_values = recent_seeds['SeedValue'].values
            trend = np.polyfit(seasons, seed_values, 1)[0]
        else:
            trend = 0.0
        
        # 计算最近种子（最近3年的平均）
        recent_avg_seed = recent_seeds['SeedValue'].mean() if len(recent_seeds) > 0 else avg_seed
        
        return {
            'avg_seed': avg_seed,
            'trend': trend,
            'recent_avg_seed': recent_avg_seed,
            'appearances': len(team_seeds)
        }
    
    # 获取两队的种子历史
    team1_history = get_team_seed_history(team1_id)
    team2_history = get_team_seed_history(team2_id)
    
    # 如果任一队伍没有历史数据，使用默认值
    if team1_history is None or team2_history is None:
        for key, value in default_seed_features.items():
            matchup_df[key] = value
        return matchup_df
    
    # 计算种子特征
    matchup_df.loc[:, 'team1_historical_seed'] = team1_history['avg_seed']
    matchup_df.loc[:, 'team2_historical_seed'] = team2_history['avg_seed']
    matchup_df.loc[:, 'team1_seed_trend'] = team1_history['trend']
    matchup_df.loc[:, 'team2_seed_trend'] = team2_history['trend']
    
    # 计算历史种子差异和比率
    matchup_df.loc[:, 'historical_seed_diff'] = (
        team1_history['avg_seed'] - team2_history['avg_seed']
    )
    matchup_df.loc[:, 'historical_seed_ratio'] = (
        team1_history['avg_seed'] / (team2_history['avg_seed'] + 1e-6)
    )
    
    # 计算最近种子差异和比率
    matchup_df.loc[:, 'recent_seed_diff'] = (
        team1_history['recent_avg_seed'] - team2_history['recent_avg_seed']
    )
    matchup_df.loc[:, 'recent_seed_ratio'] = (
        team1_history['recent_avg_seed'] / (team2_history['recent_avg_seed'] + 1e-6)
    )
    
    return matchup_df


=== 生成种子排名特征===


In [13]:
print("\n=== 生成完整特征矩阵 ===")

# 定义所有特征列名
feature_columns = [
    # 基础统计特征
    'WinRate_diff', 'WinRate_ratio',
    'TotalGames_diff', 'TotalGames_ratio',
    'Wins_diff', 'Wins_ratio',
    'Losses_diff', 'Losses_ratio',
    'WinStreak_diff', 'WinStreak_ratio',
    'PointsPerGame_diff', 'PointsPerGame_ratio',
    'PointsAllowed_diff', 'PointsAllowed_ratio',
    'OffensiveRating_diff', 'OffensiveRating_ratio',
    'DefensiveRating_diff', 'DefensiveRating_ratio',
    'NetRating_diff', 'NetRating_ratio',
    'PaceAdjustedOffRating_diff', 'PaceAdjustedOffRating_ratio',
    'PaceAdjustedDefRating_diff', 'PaceAdjustedDefRating_ratio',
    'FieldGoalPct_diff', 'FieldGoalPct_ratio',
    'ThreePointPct_diff', 'ThreePointPct_ratio',
    'FreeThrowPct_diff', 'FreeThrowPct_ratio',
    'ReboundRate_diff', 'ReboundRate_ratio',
    'AssistRate_diff', 'AssistRate_ratio',
    'TurnoverRate_diff', 'TurnoverRate_ratio',
    'BlockRate_diff', 'BlockRate_ratio',
    'StealRate_diff', 'StealRate_ratio',
    
    # 动量特征
    'last_5_win_rate_diff', 'last_5_win_rate_ratio',
    'last_10_win_rate_diff', 'last_10_win_rate_ratio',
    'last_15_win_rate_diff', 'last_15_win_rate_ratio',
    
    # 效率指标
    'offensive_efficiency_diff', 'offensive_efficiency_ratio',
    'pace_diff', 'pace_ratio',
    
    # 一致性评分
    'consistency_diff', 'consistency_ratio',
    
    # 历史交手特征
    'historical_win_rate',
    'total_matchups',
    'recent_matchup_win_rate',
    'head_to_head_dominance',
    
    # 种子排名特征
    'seed_diff',
    'seed_ratio',
    'team1_historical_seed',
    'team2_historical_seed',
    'seed_historical_diff'
]


def prepare_tournament_data(tournament_games, regular_season, tournament_seeds, massey_ordinals):
    """准备锦标赛数据，避免数据泄露"""
    print("开始准备特征数据...")
    all_features = []
    all_labels = []
    all_season_indices = []
    
    # 按赛季处理数据
    for season in range(2003, 2025):
        print(f"\n处理第 {season} 赛季...")
        
        # 获取本赛季的比赛
        season_tournament = tournament_games[tournament_games['Season'] == season]
        season_regular = regular_season[regular_season['Season'] == season]
        
        print(f"找到 {len(season_tournament)} 场锦标赛比赛")
        if len(season_tournament) == 0:
            continue
            
        games_processed = 0
        
        # 处理每场比赛
        for _, game in season_tournament.iterrows():
            try:
                # 随机决定队伍顺序，避免数据泄露
                if np.random.random() > 0.5:
                    team1_id = game['WTeamID']
                    team2_id = game['LTeamID']
                    label = 1
                else:
                    team1_id = game['LTeamID']
                    team2_id = game['WTeamID']
                    label = 0
                
                current_game_date = game['DayNum']
                
                # 计算球队统计数据（只使用当前比赛之前的数据）
                team1_stats = calculate_basic_team_stats(
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team1_id,
                    current_game_date
                )
                team2_stats = calculate_basic_team_stats(
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team2_id,
                    current_game_date
                )
                
                if team1_stats.empty or team2_stats.empty:
                    continue
                
                # 创建对抗特征
                matchup_features = create_matchup_features(
                    team1_stats,
                    team2_stats,
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team1_id,
                    team2_id,
                    current_game_date
                )
                
                if matchup_features is None:
                    continue
                
                # 添加动量特征
                team1_momentum = calculate_momentum_features(
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team1_id,
                    current_game_date
                )
                team2_momentum = calculate_momentum_features(
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team2_id,
                    current_game_date
                )
                
                for key in team1_momentum:
                    matchup_features[f'{key}_diff'] = team1_momentum[key] - team2_momentum[key]
                    matchup_features[f'{key}_ratio'] = team1_momentum[key] / (team2_momentum[key] + 1e-6)
                
                # 添加效率指标
                team1_efficiency = calculate_efficiency_metrics(
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team1_id,
                    current_game_date
                )
                team2_efficiency = calculate_efficiency_metrics(
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team2_id,
                    current_game_date
                )
                
                for key in ['offensive_efficiency', 'pace']:
                    matchup_features[f'{key}_diff'] = team1_efficiency[key] - team2_efficiency[key]
                    matchup_features[f'{key}_ratio'] = team1_efficiency[key] / (team2_efficiency[key] + 1e-6)
                
                # 添加一致性评分
                team1_consistency = calculate_consistency_rating(
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team1_id,
                    current_game_date
                )
                team2_consistency = calculate_consistency_rating(
                    season_regular[season_regular['DayNum'] < current_game_date],
                    team2_id,
                    current_game_date
                )
                matchup_features['consistency_diff'] = team1_consistency - team2_consistency
                matchup_features['consistency_ratio'] = team1_consistency / (team2_consistency + 1e-6)
                
                # 添加历史交手特征
                historical_features = add_historical_matchup_features(
                    matchup_features,  # 直接传入特征字典
                    pd.concat([
                        regular_season[regular_season['Season'] < season],
                        season_regular[season_regular['DayNum'] < current_game_date]
                    ]),
                    team1_id,
                    team2_id,
                    season,
                    current_game_date
                )

                # 添加种子排名特征
                seed_features = add_tournament_seed_features(
                    pd.DataFrame([matchup_features]),
                    tournament_seeds[tournament_seeds['Season'] < season],
                    team1_id,
                    team2_id,
                    season
                )
                
                for col in ['team1_historical_seed', 'team2_historical_seed',
                           'team1_seed_trend', 'team2_seed_trend',
                           'historical_seed_diff', 'historical_seed_ratio',
                           'recent_seed_diff', 'recent_seed_ratio']:
                    matchup_features[col] = seed_features[col].iloc[0]
                
                # 构建特征向量
                feature_vector = []
                for col in feature_columns:
                    feature_vector.append(matchup_features.get(col, 0.0))
                
                # 添加特征和标签
                all_features.append(feature_vector)
                all_labels.append(label)
                all_season_indices.append(season)
                
                # 创建反向特征（交换两队）
                reverse_vector = []
                for col in feature_columns:
                    if '_diff' in col:
                        reverse_vector.append(-matchup_features.get(col, 0.0))
                    elif '_ratio' in col:
                        val = matchup_features.get(col, 1.0)
                        reverse_vector.append(1/val if val != 0 else 0.0)
                    elif col in ['historical_win_rate', 'recent_matchup_win_rate']:
                        reverse_vector.append(1 - matchup_features.get(col, 0.5))
                    elif col == 'head_to_head_dominance':
                        reverse_vector.append(-matchup_features.get(col, 0.0))
                    else:
                        reverse_vector.append(matchup_features.get(col, 0.0))
                
                all_features.append(reverse_vector)
                all_labels.append(1 - label)
                all_season_indices.append(season)
                
                games_processed += 1
                if games_processed % 10 == 0:
                    print(f"已处理 {games_processed}/{len(season_tournament)} 场比赛")
                    
            except Exception as e:
                print(f"处理比赛时出错 {team1_id} vs {team2_id}: {str(e)}")
                continue
                
        print(f"完成 {season} 赛季，处理了 {games_processed} 场比赛")
    
    return np.array(all_features), np.array(all_labels), np.array(all_season_indices)

# 生成特征矩阵
print("开始生成特征矩阵...")
X, y, season_indices = prepare_tournament_data(
    tournament_results,
    regular_season_results,
    tournament_seeds,
    massey_ordinals
)

print("\n=== 特征生成完成 ===")
print(f"特征矩阵形状: {X.shape}")
print(f"标签数量: {len(y)}")
print(f"正样本比例: {np.mean(y):.2%}")

# 检查特征质量
print("\n检查特征质量...")
print("检查是否有无效值:")
print(f"NaN值数量: {np.isnan(X).sum()}")
print(f"Inf值数量: {np.isinf(X).sum()}")

if np.isnan(X).any() or np.isinf(X).any():
    print("修复无效值...")
    X = np.nan_to_num(X, nan=0, posinf=1e6, neginf=-1e6)


=== 生成完整特征矩阵 ===
开始生成特征矩阵...
开始准备特征数据...

处理第 2003 赛季...
找到 64 场锦标赛比赛
已处理 10/64 场比赛
已处理 20/64 场比赛
已处理 30/64 场比赛
已处理 40/64 场比赛
已处理 50/64 场比赛
已处理 60/64 场比赛
完成 2003 赛季，处理了 64 场比赛

处理第 2004 赛季...
找到 64 场锦标赛比赛
已处理 10/64 场比赛
已处理 20/64 场比赛
已处理 30/64 场比赛
已处理 40/64 场比赛
已处理 50/64 场比赛
已处理 60/64 场比赛
完成 2004 赛季，处理了 64 场比赛

处理第 2005 赛季...
找到 64 场锦标赛比赛
已处理 10/64 场比赛
已处理 20/64 场比赛
已处理 30/64 场比赛
已处理 40/64 场比赛
已处理 50/64 场比赛
已处理 60/64 场比赛
完成 2005 赛季，处理了 64 场比赛

处理第 2006 赛季...
找到 64 场锦标赛比赛
已处理 10/64 场比赛
已处理 20/64 场比赛
已处理 30/64 场比赛
已处理 40/64 场比赛
已处理 50/64 场比赛
已处理 60/64 场比赛
完成 2006 赛季，处理了 64 场比赛

处理第 2007 赛季...
找到 64 场锦标赛比赛
已处理 10/64 场比赛
已处理 20/64 场比赛
已处理 30/64 场比赛
已处理 40/64 场比赛
已处理 50/64 场比赛
已处理 60/64 场比赛
完成 2007 赛季，处理了 64 场比赛

处理第 2008 赛季...
找到 64 场锦标赛比赛
已处理 10/64 场比赛
已处理 20/64 场比赛
已处理 30/64 场比赛
已处理 40/64 场比赛
已处理 50/64 场比赛
已处理 60/64 场比赛
完成 2008 赛季，处理了 64 场比赛

处理第 2009 赛季...
找到 64 场锦标赛比赛
已处理 10/64 场比赛
已处理 20/64 场比赛
已处理 30/64 场比赛
已处理 40/64 场比赛
已处理 50/64 场比赛
已处理 60/64 场比赛
完成 2009 赛季，处理了 64 场比赛

处理第 2010 赛季.

In [15]:
print("\n=== 数据预处理和划分 ===")

# 检查原始数据维度
print("原始数据维度检查:")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"season_indices shape: {season_indices.shape}")

# 不需要重复season_indices，因为每个样本已经是成对的了
print("\n检查赛季分布...")
unique_seasons = sorted(np.unique(season_indices))
print("检测到的赛季:", unique_seasons)

# 检查并处理无效值
print("\n检查特征矩阵中的无效值...")
invalid_mask = np.isnan(X) | np.isinf(X)
if invalid_mask.any():
    print(f"发现 {np.sum(invalid_mask)} 个无效值，进行处理...")
    X = np.nan_to_num(X, nan=0, posinf=1e6, neginf=-1e6)

# 标准化特征
print("\n标准化特征...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 计算每个赛季的比赛数量
print("\n赛季数据统计:")
season_counts = {}
for season in unique_seasons:
    mask = season_indices == season
    count = np.sum(mask)
    season_counts[season] = count
    print(f"赛季 {season}: {count} 个样本 (实际比赛数: {count//2})")

# 选择验证集赛季
val_season = None
for potential_val_season in sorted(unique_seasons, reverse=True):
    if season_counts[potential_val_season] >= 67:  # 正常赛季应该有67场比赛
        val_season = potential_val_season
        break

if val_season is None:
    print("\n警告：未找到合适的验证集赛季，使用最近的赛季作为验证集")
    val_season = max(unique_seasons)
print(f"\n选择赛季 {val_season} 作为验证集")

# 创建训练集和验证集掩码
train_mask = (season_indices < val_season) & (season_indices != 2020)
val_mask = season_indices == val_season

# 验证掩码维度和数据
print("\n掩码维度检查:")
print(f"train_mask shape: {train_mask.shape}, True values: {np.sum(train_mask)}")
print(f"val_mask shape: {val_mask.shape}, True values: {np.sum(val_mask)}")
print(f"X_scaled shape: {X_scaled.shape}")

# 划分数据集
X_train = X_scaled[train_mask]
y_train = y[train_mask]
X_val = X_scaled[val_mask]
y_val = y[val_mask]

print("\n数据集划分结果:")
print(f"训练集形状: {X_train.shape}")
print(f"验证集形状: {X_val.shape}")
if len(y_train) > 0:
    print(f"训练集正样本比例: {np.mean(y_train):.2%}")
if len(y_val) > 0:
    print(f"验证集正样本比例: {np.mean(y_val):.2%}")

# 打印详细的数据集分布
print("\n训练集赛季分布:")
for season in sorted(unique_seasons):
    if season < val_season and season != 2020:
        mask = season_indices[train_mask] == season
        count = np.sum(mask)
        if count > 0:
            print(f"赛季 {season}: {count} 个样本 (实际比赛数: {count//2})")

print("\n验证集赛季分布:")
mask = season_indices[val_mask] == val_season
print(f"赛季 {val_season}: {np.sum(mask)} 个样本 (实际比赛数: {np.sum(mask)//2})")

# 保存处理后的数据
print("\n保存处理后的数据...")
processed_data = {
    'X_train': X_train,
    'X_val': X_val,
    'y_train': y_train,
    'y_val': y_val,
    'scaler': scaler,
    'train_seasons': [s for s in unique_seasons if s < val_season and s != 2020],
    'val_season': val_season
}

# 清理内存
del X, X_scaled
gc.collect()

print("\n数据预处理和划分完成！")


=== 数据预处理和划分 ===
原始数据维度检查:
X shape: (2764, 61)
y shape: (2764,)
season_indices shape: (2764,)

检查赛季分布...
检测到的赛季: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024]

检查特征矩阵中的无效值...

标准化特征...

赛季数据统计:
赛季 2003: 128 个样本 (实际比赛数: 64)
赛季 2004: 128 个样本 (实际比赛数: 64)
赛季 2005: 128 个样本 (实际比赛数: 64)
赛季 2006: 128 个样本 (实际比赛数: 64)
赛季 2007: 128 个样本 (实际比赛数: 64)
赛季 2008: 128 个样本 (实际比赛数: 64)
赛季 2009: 128 个样本 (实际比赛数: 64)
赛季 2010: 128 个样本 (实际比赛数: 64)
赛季 2011: 134 个样本 (实际比赛数: 67)
赛季 2012: 134 个样本 (实际比赛数: 67)
赛季 2013: 134 个样本 (实际比赛数: 67)
赛季 2014: 134 个样本 (实际比赛数: 67)
赛季 2015: 134 个样本 (实际比赛数: 67)
赛季 2016: 134 个样本 (实际比赛数: 67)
赛季 2017: 134 个样本 (实际比赛数: 67)
赛季 2018: 134 个样本 (实际比赛数: 67)
赛季 2019: 134 个样本 (实际比赛数: 67)
赛季 2021: 132 个样本 (实际比赛数: 66)
赛季 2022: 134 个样本 (实际比赛数: 67)
赛季 2023: 134 个样本 (实际比赛数: 67)
赛季 2024: 134 个样本 (实际比赛数: 67)

选择赛季 2024 作为验证集

掩码维度检查:
train_mask shape: (2764,), True values: 2630
val_mask shape: (2764,), True values: 134
X_

In [17]:
print("\n=== 训练LightGBM模型 ===")

# 定义时间序列交叉验证分割器
def time_series_split(season_indices_train, n_splits=5):
    unique_seasons = sorted(np.unique(season_indices_train))
    splits = []
    
    # 确保每个分割至少有3个赛季的数据
    min_seasons = 3
    for i in range(n_splits):
        split_idx = len(unique_seasons) - (n_splits - i)
        if split_idx >= min_seasons:
            val_season = unique_seasons[split_idx]
            train_mask = season_indices_train < val_season
            val_mask = season_indices_train == val_season
            splits.append((np.where(train_mask)[0], np.where(val_mask)[0]))
    
    return splits

# 修改目标函数以使用时间序列交叉验证
def objective(trial):
    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "num_leaves": trial.suggest_int("num_leaves", 8, 64),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 2, 20),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-3, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
        "max_bin": trial.suggest_int("max_bin", 128, 256),
        "n_jobs": 1
    }
    
    # 获取训练集中的season_indices
    train_seasons = season_indices[train_mask]
    
    # 获取时间序列交叉验证分割
    splits = time_series_split(train_seasons)
    
    scores = []
    # 对每个分割进行训练和评估
    for train_idx, val_idx in splits:
        # 创建训练集和验证集
        fold_train_data = lgb.Dataset(
            X_train[train_idx], 
            label=y_train[train_idx]
        )
        fold_val_data = lgb.Dataset(
            X_train[val_idx], 
            label=y_train[val_idx],
            reference=fold_train_data
        )
        
        # 训练模型
        model = lgb.train(
            params,
            fold_train_data,
            num_boost_round=1000,
            valid_sets=[fold_val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50),
                lgb.log_evaluation(0)
            ]
        )
        
        # 直接添加最佳验证分数
        scores.append(model.best_score['valid_0']['binary_logloss'])
    
    # 返回平均分数
    return np.mean(scores)

# 运行参数优化
print("开始参数优化...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

# 获取最佳参数
best_params = study.best_params
best_params.update({
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "n_jobs": 1
})

print("\n最佳参数:")
for key, value in best_params.items():
    print(f"{key}: {value}")

# 使用最佳参数训练最终模型
print("\n使用最佳参数训练最终模型...")
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=100)
]

lgb_model = lgb.train(
    best_params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    callbacks=callbacks
)

# 评估模型
lgb_pred = lgb_model.predict(X_val)
lgb_brier_score = np.mean((lgb_pred - y_val) ** 2)
print(f"\nLightGBM Brier Score: {lgb_brier_score:.4f}")

# 特征重要性分析
importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': lgb_model.feature_importance()
})
importance = importance.sort_values('importance', ascending=False)
print("\n前10个最重要的特征:")
print(importance.head(10))

[I 2025-02-21 06:54:55,796] A new study created in memory with name: no-name-5450c83d-a9de-4ed6-9cfc-eb87b7b64018



=== 训练LightGBM模型 ===
开始参数优化...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[17]	valid_0's binary_logloss: 0.636668
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[58]	valid_0's binary_logloss: 0.531818
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[27]	valid_0's binary_logloss: 0.615188
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[14]	valid_0's binary_logloss: 0.659975
Training until validation scores don't improve for 50 rounds


[I 2025-02-21 06:55:00,239] Trial 0 finished with value: 0.6080756371211906 and parameters: {'num_leaves': 62, 'learning_rate': 0.05484819445087828, 'feature_fraction': 0.9803488873657714, 'bagging_fraction': 0.7361581119171194, 'min_child_samples': 16, 'min_child_weight': 0.6081202203202107, 'reg_alpha': 5.002248037451428e-08, 'reg_lambda': 0.05952379967729062, 'max_bin': 183}. Best is trial 0 with value: 0.6080756371211906.


Early stopping, best iteration is:
[65]	valid_0's binary_logloss: 0.596728
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[36]	valid_0's binary_logloss: 0.615846
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[102]	valid_0's binary_logloss: 0.536143
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[67]	valid_0's binary_logloss: 0.611191
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[9]	valid_0's binary_logloss: 0.668758
Training until validation scores don't improve for 50 rounds


[I 2025-02-21 06:55:03,542] Trial 1 finished with value: 0.611668373969756 and parameters: {'num_leaves': 26, 'learning_rate': 0.04246569224992203, 'feature_fraction': 0.9514421681737039, 'bagging_fraction': 0.9877309689854143, 'min_child_samples': 10, 'min_child_weight': 0.6971849535655913, 'reg_alpha': 0.0023562603075453544, 'reg_lambda': 0.002594115584854894, 'max_bin': 238}. Best is trial 0 with value: 0.6080756371211906.


Early stopping, best iteration is:
[63]	valid_0's binary_logloss: 0.626403
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[29]	valid_0's binary_logloss: 0.620638
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[33]	valid_0's binary_logloss: 0.533938
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[33]	valid_0's binary_logloss: 0.591731
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's binary_logloss: 0.665478
Training until validation scores don't improve for 50 rounds


... (32 outputs left out)

[I 2025-02-21 06:55:58,272] Trial 18 finished with value: 0.6082912543200322 and parameters: {'num_leaves': 55, 'learning_rate': 0.08104470054224149, 'feature_fraction': 0.7104064220142944, 'bagging_fraction': 0.8079509278658157, 'min_child_samples': 13, 'min_child_weight': 0.7867746939997314, 'reg_alpha': 2.7171184206332504e-05, 'reg_lambda': 0.6774662002531897, 'max_bin': 163}. Best is trial 2 with value: 0.6042971702230974.


Early stopping, best iteration is:
[36]	valid_0's binary_logloss: 0.603707
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[17]	valid_0's binary_logloss: 0.628627
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[60]	valid_0's binary_logloss: 0.5303
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[34]	valid_0's binary_logloss: 0.602122
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[10]	valid_0's binary_logloss: 0.658306
Training until validation scores don't improve for 50 rounds


[I 2025-02-21 06:56:01,275] Trial 19 finished with value: 0.6050300118238147 and parameters: {'num_leaves': 49, 'learning_rate': 0.06603628139919139, 'feature_fraction': 0.8267005579044584, 'bagging_fraction': 0.9518350814656632, 'min_child_samples': 18, 'min_child_weight': 0.15576395698827789, 'reg_alpha': 0.06049842894677693, 'reg_lambda': 0.06587980306483532, 'max_bin': 186}. Best is trial 2 with value: 0.6042971702230974.


Early stopping, best iteration is:
[21]	valid_0's binary_logloss: 0.605795

最佳参数:
num_leaves: 54
learning_rate: 0.06361107226811433
feature_fraction: 0.8266429803253279
bagging_fraction: 0.908381133674399
min_child_samples: 16
min_child_weight: 0.7148721564576287
reg_alpha: 3.942069321855748e-07
reg_lambda: 0.6217628297700628
max_bin: 134
objective: binary
metric: binary_logloss
boosting_type: gbdt
verbosity: -1
n_jobs: 1

使用最佳参数训练最终模型...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[43]	training's binary_logloss: 0.341671	valid_1's binary_logloss: 0.568921

LightGBM Brier Score: 0.1922

前10个最重要的特征:
                   feature  importance
18          NetRating_diff         114
45  last_15_win_rate_ratio         104
58   team1_historical_seed         102
37         BlockRate_ratio         102
59   team2_historical_seed          87
30        ReboundRate_diff          84
14    OffensiveRating_diff          77
36          BlockRate_diff    